# ML3. Hyperparameter Optimization / Оптимизация гиперпараметров

Решение 7 части задания проекта ML3.

**Задача:** подобрать гиперпараметры `alpha` и `l1_ratio` для `sklearn.linear_model.ElasticNet`, обучить итоговые модели и сравнить подходы.

**Target:** `price`.

**Датасет:** `src/data/train_df_processed_full.csv`.

## Что требуется по заданию

1. Реализовать собственные методы `grid search` и `random search` для `alpha` и `l1_ratio` модели `ElasticNet`.
2. Найти лучшую комбинацию гиперпараметров.
3. Обучить модель с найденными гиперпараметрами.
4. Импортировать `optuna` и настроить такой же эксперимент с `ElasticNet`.
5. Оценить метрики и сравнить подходы.
6. Запустить `optuna` на одной из схем cross-validation.

## Импорты и настройки

Для честной оптимизации используется разделение данных на `train/validation/test` по полю `created`. На `train` модель обучается, на `validation` подбираются гиперпараметры, а `test` используется только для финальной проверки качества.

In [ ]:
from pathlib import Path
import sys
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import optuna
except ImportError as error:
    raise ImportError('Для выполнения задания 7 нужна библиотека optuna. Установите ее через pip install optuna.') from error

RANDOM_STATE = 21
TARGET = 'price'
DATE_FIELD = 'created'

current_dir = Path.cwd()
if current_dir.name == 'src':
    src_dir = current_dir
    project_root = current_dir.parent
else:
    project_root = current_dir
    src_dir = project_root / 'src'

if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

DATA_PATH = src_dir / 'data' / 'train_df_processed_full.csv'

from ml3_03_data_split_methods_jacquieb import split_by_date_ratio_train_validation_test
from ml3_04_cv_methods_jacquieb import make_time_series_splits

sns.set_style('darkgrid')
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Загрузка и разбиение данных

Используем разбиение 60/20/20 по дате:

- `train` — обучение моделей;
- `validation` — подбор гиперпараметров;
- `test` — финальное сравнение подходов.

In [ ]:
df = pd.read_csv(DATA_PATH)
df[DATE_FIELD] = pd.to_datetime(df[DATE_FIELD])

df_train, df_validation, df_test = split_by_date_ratio_train_validation_test(
    df,
    validation_size=0.2,
    test_size=0.2,
    date_field=DATE_FIELD,
    random_state=RANDOM_STATE,
)

for name, part in [('train', df_train), ('validation', df_validation), ('test', df_test)]:
    print(
        name,
        part.shape,
        'доля:', round(len(part) / len(df), 3),
        'даты:', part[DATE_FIELD].min(), '-', part[DATE_FIELD].max(),
    )

## Подготовка признаков

Исключаем `price`, потому что это target. Поле `created` используется для временного разбиения и не передается в модель как числовой признак. Поле `interest` также исключаем: в этой задаче предсказывается именно `price`, а `interest` является отдельной целевой/оценочной переменной из исходной постановки.

In [ ]:
excluded_columns = {TARGET, DATE_FIELD, 'interest'}
numeric_columns = df_train.select_dtypes(include=[np.number]).columns
feature_columns = [column for column in numeric_columns if column not in excluded_columns]

X_train = df_train[feature_columns].replace([np.inf, -np.inf], np.nan)
y_train = df_train[TARGET]

X_validation = df_validation[feature_columns].replace([np.inf, -np.inf], np.nan)
y_validation = df_validation[TARGET]

X_test = df_test[feature_columns].replace([np.inf, -np.inf], np.nan)
y_test = df_test[TARGET]

print('Количество признаков:', len(feature_columns))
print(feature_columns)

## Вспомогательные функции

`ElasticNet` совмещает L1- и L2-регуляризацию:

- `alpha` задает общую силу регуляризации;
- `l1_ratio` задает баланс между L1 и L2: при `l1_ratio=1` модель близка к Lasso, при `l1_ratio=0` — к Ridge.

Так как регуляризация чувствительна к масштабу признаков, перед моделью используем `MinMaxScaler`.

In [ ]:
def make_elastic_net_model(alpha, l1_ratio):
    return make_pipeline(
        SimpleImputer(strategy='median'),
        MinMaxScaler(),
        ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=100_000,
            random_state=RANDOM_STATE,
        ),
    )


def calculate_metrics(model, X, y):
    predictions = model.predict(X)
    return {
        'MAE': mean_absolute_error(y, predictions),
        'RMSE': np.sqrt(mean_squared_error(y, predictions)),
        'R2': r2_score(y, predictions),
    }


def validation_rmse(alpha, l1_ratio):
    model = make_elastic_net_model(alpha=alpha, l1_ratio=l1_ratio)
    model.fit(X_train, y_train)
    return calculate_metrics(model, X_validation, y_validation)['RMSE']


def fit_final_model(method_name, alpha, l1_ratio, search_time_sec, cv_rmse=None):
    model = make_elastic_net_model(alpha=alpha, l1_ratio=l1_ratio)

    start_time = perf_counter()
    model.fit(X_train, y_train)
    fit_time_sec = perf_counter() - start_time

    row = {
        'method': method_name,
        'alpha': alpha,
        'l1_ratio': l1_ratio,
        'search_time_sec': search_time_sec,
        'fit_time_sec': fit_time_sec,
        'total_time_sec': search_time_sec + fit_time_sec,
        'cv_rmse': cv_rmse,
    }

    for sample_name, X_part, y_part in [
        ('train', X_train, y_train),
        ('validation', X_validation, y_validation),
        ('test', X_test, y_test),
    ]:
        for metric_name, metric_value in calculate_metrics(model, X_part, y_part).items():
            row[f'{sample_name}_{metric_name}'] = metric_value

    return model, row

## 1. Собственная реализация Grid Search

Grid Search перебирает все пары значений из заданной сетки. Метод простой и детерминированный, но количество обучений быстро растет при увеличении числа параметров и вариантов.

In [ ]:
def grid_search_elastic_net(alpha_values, l1_ratio_values):
    rows = []
    start_time = perf_counter()

    for alpha in alpha_values:
        for l1_ratio in l1_ratio_values:
            rmse = validation_rmse(alpha=alpha, l1_ratio=l1_ratio)
            rows.append({
                'alpha': alpha,
                'l1_ratio': l1_ratio,
                'validation_RMSE': rmse,
            })

    elapsed_time = perf_counter() - start_time
    results = pd.DataFrame(rows).sort_values('validation_RMSE').reset_index(drop=True)
    best_params = results.iloc[0][['alpha', 'l1_ratio']].to_dict()

    return results, best_params, elapsed_time


alpha_grid = [0.001, 0.01, 0.1, 1.0, 10.0]
l1_ratio_grid = [0.05, 0.25, 0.5, 0.75, 1.0]

grid_results, grid_best_params, grid_search_time = grid_search_elastic_net(
    alpha_values=alpha_grid,
    l1_ratio_values=l1_ratio_grid,
)

print('Лучшие параметры Grid Search:', grid_best_params)
display(grid_results.head(10))

## 2. Собственная реализация Random Search

Random Search не перебирает полную сетку, а случайно выбирает комбинации гиперпараметров. Для `alpha` используем логарифмическую шкалу, потому что сила регуляризации обычно меняется на порядки.

In [ ]:
def random_search_elastic_net(n_iter=25, alpha_min=1e-3, alpha_max=100.0, l1_ratio_min=0.05, l1_ratio_max=1.0):
    rng = np.random.default_rng(RANDOM_STATE)
    rows = []
    start_time = perf_counter()

    for iteration in range(1, n_iter + 1):
        alpha = 10 ** rng.uniform(np.log10(alpha_min), np.log10(alpha_max))
        l1_ratio = rng.uniform(l1_ratio_min, l1_ratio_max)
        rmse = validation_rmse(alpha=alpha, l1_ratio=l1_ratio)

        rows.append({
            'iteration': iteration,
            'alpha': alpha,
            'l1_ratio': l1_ratio,
            'validation_RMSE': rmse,
        })

    elapsed_time = perf_counter() - start_time
    results = pd.DataFrame(rows).sort_values('validation_RMSE').reset_index(drop=True)
    best_params = results.iloc[0][['alpha', 'l1_ratio']].to_dict()

    return results, best_params, elapsed_time


random_results, random_best_params, random_search_time = random_search_elastic_net(n_iter=25)

print('Лучшие параметры Random Search:', random_best_params)
display(random_results.head(10))

## 3. Optuna на train/validation

`Optuna` использует последовательную оптимизацию: следующие пробы выбираются с учетом результатов предыдущих. В отличие от обычного случайного перебора, Bayesian/TPE-подход быстрее концентрируется на перспективных областях пространства гиперпараметров.

По чеклисту важно показать, что Optuna приближается к оптимальным параметрам за меньшее число операций. Поэтому для holdout-эксперимента задаем `n_trials=10`, тогда как Grid Search проверяет `5 * 5 = 25` комбинаций.

In [ ]:
def optuna_objective(trial):
    alpha = trial.suggest_float('alpha', 1e-3, 100.0, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.05, 1.0)
    return validation_rmse(alpha=alpha, l1_ratio=l1_ratio)


start_time = perf_counter()
optuna_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
optuna_study.optimize(optuna_objective, n_trials=10, show_progress_bar=False)
optuna_search_time = perf_counter() - start_time

optuna_results = optuna_study.trials_dataframe().sort_values('value').reset_index(drop=True)
optuna_best_params = optuna_study.best_params

print('Лучшие параметры Optuna:', optuna_best_params)
print('Лучший validation RMSE:', optuna_study.best_value)
display(optuna_results.head(10))

## Сравнение количества операций

Под операцией здесь понимаем одно обучение модели с одной комбинацией `alpha` и `l1_ratio` на схеме `train -> validation`. Для Grid Search это число всех комбинаций сетки, для Random Search — число случайных проб, для Optuna — число trials.

Такой блок нужен для проверки требования чеклиста: Optuna должна приблизиться к оптимальным параметрам за меньшее количество операций.

In [ ]:
operations_comparison = pd.DataFrame([
    {
        'method': 'Grid Search',
        'n_operations': len(alpha_grid) * len(l1_ratio_grid),
        'best_validation_RMSE': grid_results['validation_RMSE'].min(),
    },
    {
        'method': 'Random Search',
        'n_operations': len(random_results),
        'best_validation_RMSE': random_results['validation_RMSE'].min(),
    },
    {
        'method': 'Optuna holdout',
        'n_operations': len(optuna_study.trials),
        'best_validation_RMSE': optuna_study.best_value,
    },
])

operations_comparison['rmse_difference_from_grid'] = (
    operations_comparison['best_validation_RMSE'] - grid_results['validation_RMSE'].min()
)
operations_comparison['rmse_difference_from_grid_pct'] = (
    operations_comparison['rmse_difference_from_grid'] / grid_results['validation_RMSE'].min() * 100
)

display(operations_comparison.sort_values('best_validation_RMSE'))

grid_operations = len(alpha_grid) * len(l1_ratio_grid)
optuna_operations = len(optuna_study.trials)

print(f'Grid Search проверил комбинаций: {grid_operations}')
print(f'Optuna проверила trials: {optuna_operations}')
print('Optuna выполнила меньше операций:', optuna_operations < grid_operations)

## 4. Обучение итоговых моделей

Теперь обучаем по одной итоговой модели для каждого подхода с найденными гиперпараметрами и сравниваем метрики на `train`, `validation` и `test`.

In [ ]:
grid_model, grid_row = fit_final_model(
    method_name='Grid Search',
    alpha=grid_best_params['alpha'],
    l1_ratio=grid_best_params['l1_ratio'],
    search_time_sec=grid_search_time,
)

random_model, random_row = fit_final_model(
    method_name='Random Search',
    alpha=random_best_params['alpha'],
    l1_ratio=random_best_params['l1_ratio'],
    search_time_sec=random_search_time,
)

optuna_model, optuna_row = fit_final_model(
    method_name='Optuna holdout',
    alpha=optuna_best_params['alpha'],
    l1_ratio=optuna_best_params['l1_ratio'],
    search_time_sec=optuna_search_time,
)

holdout_comparison = pd.DataFrame([grid_row, random_row, optuna_row]).sort_values('validation_RMSE').reset_index(drop=True)
display(holdout_comparison)

## 5. Optuna на cross-validation

Для CV-эксперимента используем time series split из ранее реализованного файла `ml3_04_cv_methods_jacquieb.py`. Это логично для текущего датасета, потому что есть поле `created`, и модель должна проверяться на более поздних временных блоках.

In [ ]:
cv_splits = make_time_series_splits(df_train, k=4, date_field=DATE_FIELD)
print('Количество CV-разбиений:', len(cv_splits))

for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
    print(f'fold {fold}: train={len(train_idx)}, validation={len(valid_idx)}')

In [ ]:
def cv_rmse_for_params(alpha, l1_ratio, splits):
    fold_scores = []

    for train_idx, valid_idx in splits:
        fold_train = df_train.iloc[train_idx]
        fold_valid = df_train.iloc[valid_idx]

        X_fold_train = fold_train[feature_columns].replace([np.inf, -np.inf], np.nan)
        y_fold_train = fold_train[TARGET]
        X_fold_valid = fold_valid[feature_columns].replace([np.inf, -np.inf], np.nan)
        y_fold_valid = fold_valid[TARGET]

        model = make_elastic_net_model(alpha=alpha, l1_ratio=l1_ratio)
        model.fit(X_fold_train, y_fold_train)
        fold_scores.append(calculate_metrics(model, X_fold_valid, y_fold_valid)['RMSE'])

    return float(np.mean(fold_scores))


def optuna_cv_objective(trial):
    alpha = trial.suggest_float('alpha', 1e-3, 100.0, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.05, 1.0)
    return cv_rmse_for_params(alpha=alpha, l1_ratio=l1_ratio, splits=cv_splits)


start_time = perf_counter()
optuna_cv_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
optuna_cv_study.optimize(optuna_cv_objective, n_trials=20, show_progress_bar=False)
optuna_cv_search_time = perf_counter() - start_time

optuna_cv_best_params = optuna_cv_study.best_params
optuna_cv_results = optuna_cv_study.trials_dataframe().sort_values('value').reset_index(drop=True)

print('Лучшие параметры Optuna CV:', optuna_cv_best_params)
print('Лучший средний CV RMSE:', optuna_cv_study.best_value)
display(optuna_cv_results.head(10))

## 6. Сравнение всех подходов

Модель `Optuna CV` обучаем с найденными на cross-validation гиперпараметрами и оцениваем на тех же трех частях датасета, чтобы таблица была сопоставима с остальными подходами.

In [ ]:
optuna_cv_model, optuna_cv_row = fit_final_model(
    method_name='Optuna time series CV',
    alpha=optuna_cv_best_params['alpha'],
    l1_ratio=optuna_cv_best_params['l1_ratio'],
    search_time_sec=optuna_cv_search_time,
    cv_rmse=optuna_cv_study.best_value,
)

comparison = pd.DataFrame([grid_row, random_row, optuna_row, optuna_cv_row])
comparison = comparison.sort_values('validation_RMSE').reset_index(drop=True)
display(comparison)

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=comparison, x='validation_RMSE', y='method', color='steelblue')
plt.title('Сравнение подходов по RMSE на validation')
plt.xlabel('Validation RMSE')
plt.ylabel('')
plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(data=comparison, x='test_RMSE', y='method', color='seagreen')
plt.title('Сравнение подходов по RMSE на test')
plt.xlabel('Test RMSE')
plt.ylabel('')
plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(data=comparison, x='total_time_sec', y='method', color='darkorange')
plt.title('Сравнение подходов по времени поиска и обучения')
plt.xlabel('Секунды')
plt.ylabel('')
plt.show()

## Вывод

`Grid Search` прост и полностью проверяет заданную сетку, но качество зависит от того, насколько удачно эта сетка выбрана.

`Random Search` исследует больше разных значений `alpha`, особенно на логарифмической шкале, но результат зависит от случайной выборки параметров.

`Optuna` обычно эффективнее, потому что следующие пробы выбираются с учетом предыдущих результатов. В holdout-варианте она оптимизирует качество на `validation`, а в CV-варианте оценивает параметры устойчивее, потому что использует несколько временных разбиений.

В таблице `operations_comparison` видно требование из чеклиста: `Grid Search` проверяет 25 комбинаций, а `Optuna holdout` использует 10 trials и получает близкое значение `validation_RMSE`.

Итоговый выбор лучше делать по `validation_RMSE` и `test_RMSE`, одновременно учитывая время поиска. Если Optuna CV дает близкое качество на `test`, ее можно считать наиболее надежной схемой, потому что гиперпараметры проверялись не на одном разбиении, а на нескольких временных fold.